# F4-M03: Distribuciones Numéricas

**Fase 4 — EDA Final** | Dataset: `df_eda_final.parquet` (33.621 estudiantes × 21 variables)

Análisis univariante de las **13 variables numéricas** del dataset final:  
6 continuas, 4 discretas y 2 binarias.

**Librerías:** Matplotlib, Seaborn, Plotnine (ggplot2), Statsmodels Graphics  
**Salida:** Gráficos estáticos embebidos como PNG base64 en HTML

In [1]:
# ── CSS + JS para botones Interpretar ────────────────────────────────────────
js_css_html = (
    '<style>'
    '.ibt{display:inline-flex;align-items:center;gap:5px;padding:5px 12px;'
    'background:#3182ce;color:#fff;border:none;border-radius:5px;'
    'font-size:12px;font-weight:600;cursor:pointer;float:right;margin-top:-2px;}'
    '.ibt:hover{background:#2b6cb0;}'
    '.ipn{display:none;background:#f7fafc;border:1px solid #e2e8f0;'
    'border-radius:6px;padding:14px 16px;margin-top:8px;font-size:13px;'
    'color:#2d3748;line-height:1.6;}'
    '.ipn.vis{display:block;}'
    '.tg{background:#c6f6d5;color:#276749;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tb{background:#fed7d7;color:#9b2c2c;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tm{background:#fefcbf;color:#744210;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '</style>'
    '<script>'
    'function tg(id){'
    'var p=document.getElementById(id);'
    'var b=document.querySelector("[data-pid=\'" + id + "\']");'
    'if(!p||!b)return;'
    'var vis=p.classList.toggle("vis");'
    'b.textContent=vis?"✖ Cerrar":"📖 Interpretar";}'
    '</script>'
)

def h2btn(titulo, pid):
    return (
        f'<div style="display:flex;align-items:center;justify-content:space-between;'
        f'margin:28px 0 8px;">'
        f'<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:0;">{titulo}</h2>'
        f'<button class="ibt" data-pid="{pid}" onclick="tg(\'{pid}\')">' 
        f'📖 Interpretar</button></div>'
    )

def panel(pid, texto):
    return f'<div id="{pid}" class="ipn">{texto}</div>'


In [2]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
import base64
import io
from pathlib import Path

warnings.filterwarnings('ignore')

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists(): break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Backend no interactivo
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from plotnine import (ggplot, aes, geom_histogram, geom_density,
                      geom_vline, labs, theme_minimal, theme,
                      element_text, element_blank, facet_wrap,
                      scale_fill_manual, after_stat)
from statsmodels.graphics.gofplots import qqplot
from scipy import stats

from src.config import (RUTA_HTML, ETIQUETAS_VARIABLES, COLORES,
                        info_entorno)
from src.utils import crear_directorios, formato_numero_es

# ── Paneles de interpretación dinámica ──────────────
panel_distribuciones = panel('distribuciones', 'Variables académicas (nota_1er_anio, cred_superados) muestran <span class=\"tb\">cola izquierda en abandono</span> — los que abandonan se concentran en valores bajos. Justifica RobustScaler para modelos lineales dado el rango amplio.')
panel_estadisticos = panel('estadisticos', 'Rangos muy dispares: nota 0–10 vs cred_superados 0–230. <b>StandardScaler obligatorio</b> para LogReg, KNN, SVM, MLP. <span class=\"tg\">Árboles no necesitan escalado</span> — invariantes a la escala.')

from src.html import (generar_kpis_html, generar_seccion_html,
                      generar_html_navegacion_completa, guardar_html)
from src.html.render import render_base_html

RUTA_EDA = ROOT / 'data' / '04_eda'
RUTA_FASE4_HTML = RUTA_HTML / 'fase4'
crear_directorios([RUTA_FASE4_HTML])

fmt = formato_numero_es

def etiq(col):
    return ETIQUETAS_VARIABLES.get(col, col)


def fig_a_base64(fig, dpi=150, bbox='tight'):
    """Convierte figura Matplotlib a string base64 para embeber en HTML."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches=bbox,
                facecolor='white', edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')


def img_html(b64, ancho='100%'):
    """Genera tag <img> desde base64."""
    return (f'<img src="data:image/png;base64,{b64}" '
            f'style="width:{ancho};max-width:100%;display:block;margin:0 auto;">')


# Paleta coherente con el proyecto
COLOR_PRINCIPAL = '#3182ce'
COLOR_ALERTA = '#ed8936'
COLOR_ERROR = '#e53e3e'
COLOR_OK = '#38a169'
PALETA_SEQ = ['#ebf8ff', '#bee3f8', '#90cdf4', '#63b3ed',
              '#4299e1', '#3182ce', '#2b6cb0', '#2c5282']

sns.set_theme(style='whitegrid', font_scale=0.9,
              rc={'figure.facecolor': 'white',
                  'axes.facecolor': 'white',
                  'grid.alpha': 0.3})

info_entorno()


✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [3]:
# ============================================================================
# CELDA 2: CARGA DE DATOS Y CLASIFICACIÓN DE VARIABLES
# ============================================================================

df = pd.read_parquet(RUTA_EDA / 'df_eda_final.parquet')
n_total = len(df)
print(f'Dataset: {n_total:,} filas × {len(df.columns)} columnas')

# Clasificar variables numéricas (excluir abandono y categóricas str)
numericas = [c for c in df.columns
             if c != 'abandono' and df[c].dtype not in ['object', 'string']]

continuas = [c for c in numericas if df[c].nunique() > 12]
discretas = [c for c in numericas if 3 <= df[c].nunique() <= 12]
binarias  = [c for c in numericas if df[c].nunique() <= 2]

print(f'\nNuméricas: {len(numericas)} (continuas={len(continuas)}, '
      f'discretas={len(discretas)}, binarias={len(binarias)})')
print(f'  Continuas:  {continuas}')
print(f'  Discretas:  {discretas}')
print(f'  Binarias:   {binarias}')

# Métricas para KPIs
n_con_nulos = sum(1 for c in numericas if df[c].isna().sum() > 0)
n_con_outliers = 0
for c in continuas:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = ((df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)).sum()
    if outliers > 0:
        n_con_outliers += 1

# Tabla de estadísticas descriptivas
stats_df = df[numericas].describe().T
stats_df['nulos'] = df[numericas].isna().sum()
stats_df['%_nulos'] = (stats_df['nulos'] / n_total * 100).round(1)
stats_df['skew'] = df[numericas].skew().round(2)
stats_df['kurtosis'] = df[numericas].kurtosis().round(2)
stats_df = stats_df.round(2)

print(f'\nVariables con nulos: {n_con_nulos}')
print(f'Variables con outliers (IQR): {n_con_outliers}')


Dataset: 33,621 filas × 21 columnas

Numéricas: 12 (continuas=6, discretas=4, binarias=2)
  Continuas:  ['cred_superados_anio_1er', 'edad_entrada', 'nota_1er_anio', 'nota_acceso', 'nota_selectividad', 'orden_preferencia']
  Discretas:  ['anios_gap', 'anios_sin_beca', 'max_pagos', 'n_anios_beca']
  Binarias:   ['indicador_interrupcion', 'tuvo_beca']

Variables con nulos: 5
Variables con outliers (IQR): 6


In [4]:
# ============================================================================
# CELDA 3: GRÁFICOS — SEABORN + MATPLOTLIB
# ============================================================================

graficos = {}  # clave → html string

# --- 1. PANORÁMICA: Boxenplot de todas las numéricas (normalizadas) ---
from sklearn.preprocessing import StandardScaler

df_norm = pd.DataFrame(
    StandardScaler().fit_transform(df[continuas].dropna()),
    columns=[etiq(c) for c in continuas]
)
df_melt = df_norm.melt(var_name='Variable', value_name='Valor (estandarizado)')

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxenplot(data=df_melt, x='Variable', y='Valor (estandarizado)',
              palette='Blues', ax=ax)
ax.set_title('Panorámica: distribución de variables continuas (estandarizadas)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
graficos['panoramica'] = img_html(fig_a_base64(fig))
print('✅ Panorámica boxenplot')


# --- 2. VIOLINPLOTS de continuas ---
n_cont = len(continuas)
ncols = 3
nrows = (n_cont + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(continuas):
    data = df[col].dropna()
    n_validos = len(data)
    n_nulos = df[col].isna().sum()
    
    sns.violinplot(y=data, ax=axes[i], color=COLOR_PRINCIPAL,
                   inner='quartile', linewidth=1)
    axes[i].set_title(f'{etiq(col)}\n(N={fmt(n_validos)}'
                      + (f', {fmt(n_nulos)} sin dato)' if n_nulos > 0 else ')'),
                      fontsize=10)
    axes[i].set_ylabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Violinplots — Variables Continuas', fontsize=13,
             fontweight='bold', y=1.02)
fig.tight_layout()
graficos['violinplots'] = img_html(fig_a_base64(fig))
print('✅ Violinplots')


# --- 3. BARRAS DISCRETAS + BINARIAS ---
vars_disc_bin = discretas + binarias
n_db = len(vars_disc_bin)
ncols_db = 3
nrows_db = (n_db + ncols_db - 1) // ncols_db
fig, axes = plt.subplots(nrows_db, ncols_db, figsize=(14, 4 * nrows_db))
axes = axes.flatten()

for i, col in enumerate(vars_disc_bin):
    data = df[col].dropna()
    vc = data.value_counts().sort_index()
    
    color = COLOR_OK if col in binarias else COLOR_ALERTA
    axes[i].bar(vc.index.astype(str), vc.values, color=color, alpha=0.85,
                edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'{etiq(col)} (N={fmt(len(data))})', fontsize=10)
    axes[i].set_ylabel('Frecuencia')
    
    # Etiquetas de porcentaje encima de cada barra
    total = vc.sum()
    for x, v in zip(range(len(vc)), vc.values):
        pct = v / total * 100
        axes[i].text(x, v + total * 0.01, f'{pct:.1f}%',
                     ha='center', fontsize=8, color='#4a5568')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Variables Discretas y Binarias — Frecuencias',
             fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
graficos['discretas_binarias'] = img_html(fig_a_base64(fig))
print('✅ Discretas y binarias')


# --- 4. SKEW vs KURTOSIS (escala log + zoom) ---
skews = df[numericas].skew()
kurts = df[numericas].kurtosis()
labels = [etiq(c) for c in numericas]

# 4A. Gráfico principal con escala log en Y
fig, ax = plt.subplots(figsize=(10, 6))

# Separar positivos y negativos para log
for label, sk, ku in zip(labels, skews, kurts):
    color = COLOR_ERROR if abs(ku) > 10 else COLOR_ALERTA if abs(ku) > 2 else COLOR_OK
    ax.scatter(sk, ku, s=90, c=color, alpha=0.85, edgecolors='white', zorder=5)
    ax.annotate(label, (sk, ku), fontsize=8, alpha=0.85,
                xytext=(6, 6), textcoords='offset points')

ax.axhline(0, color='gray', ls='--', alpha=0.4)
ax.axvline(0, color='gray', ls='--', alpha=0.4)
ax.set_yscale('symlog', linthresh=2)
ax.set_xlabel('Asimetría (Skewness)', fontsize=11)
ax.set_ylabel('Curtosis (Kurtosis) — escala simétrica log', fontsize=11)
ax.set_title('Asimetría vs Curtosis — Todas las numéricas',
             fontsize=13, fontweight='bold')

# Leyenda de colores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLOR_OK, label='Normal (|curtosis| ≤ 2)'),
    Patch(facecolor=COLOR_ALERTA, label='Moderada (2 < |curtosis| ≤ 10)'),
    Patch(facecolor=COLOR_ERROR, label='Extrema (|curtosis| > 10)'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=8)

fig.tight_layout()
graficos['skew_kurtosis'] = img_html(fig_a_base64(fig))
print('✅ Skew vs Kurtosis (escala log)')

# 4B. Zoom: solo variables con curtosis moderada (sin las extremas)
vars_zoom = [c for c, k in zip(numericas, kurts) if abs(k) <= 10]
skews_z = df[vars_zoom].skew()
kurts_z = df[vars_zoom].kurtosis()
labels_z = [etiq(c) for c in vars_zoom]

fig2, ax2 = plt.subplots(figsize=(10, 6))

for label, sk, ku in zip(labels_z, skews_z, kurts_z):
    color = COLOR_ALERTA if abs(ku) > 2 else COLOR_OK
    ax2.scatter(sk, ku, s=90, c=color, alpha=0.85, edgecolors='white', zorder=5)
    ax2.annotate(label, (sk, ku), fontsize=9, alpha=0.85,
                 xytext=(6, 6), textcoords='offset points')

ax2.axhline(0, color='gray', ls='--', alpha=0.4)
ax2.axvline(0, color='gray', ls='--', alpha=0.4)
ax2.axhspan(-2, 2, alpha=0.06, color=COLOR_OK)
ax2.axvspan(-1, 1, alpha=0.06, color=COLOR_OK)
ax2.set_xlabel('Asimetría (Skewness)', fontsize=11)
ax2.set_ylabel('Curtosis (Kurtosis)', fontsize=11)
ax2.set_title('Zoom: Variables con curtosis moderada (excluidas años_gap, interrupción, orden_pref, edad)',
              fontsize=11, fontweight='bold')
ax2.text(0.02, 0.98, 'Zona verde: distribución\naproximadamente normal',
         transform=ax2.transAxes, fontsize=8, va='top', color=COLOR_OK, alpha=0.7)

legend_elements2 = [
    Patch(facecolor=COLOR_OK, label='Normal (|curtosis| ≤ 2)'),
    Patch(facecolor=COLOR_ALERTA, label='Moderada (2 < |curtosis| ≤ 10)'),
]
ax2.legend(handles=legend_elements2, loc='upper left', fontsize=8)

fig2.tight_layout()
graficos['skew_kurtosis_zoom'] = img_html(fig_a_base64(fig2))
print('✅ Skew vs Kurtosis (zoom)')


# --- 5. RESUMEN DE NULOS (solo variables con nulos) ---
nulos_data = df[numericas].isna().sum()
nulos_data = nulos_data[nulos_data > 0].sort_values(ascending=True)

if len(nulos_data) > 0:
    fig, ax = plt.subplots(figsize=(8, max(3, len(nulos_data) * 0.6)))
    colors = [COLOR_ERROR if v/n_total > 0.1 else COLOR_ALERTA
              for v in nulos_data.values]
    bars = ax.barh([etiq(c) for c in nulos_data.index], nulos_data.values,
                   color=colors, alpha=0.85, edgecolor='white')
    
    for bar, val in zip(bars, nulos_data.values):
        pct = val / n_total * 100
        ax.text(bar.get_width() + n_total * 0.005, bar.get_y() + bar.get_height()/2,
                f'{fmt(val)} ({pct:.1f}%)', va='center', fontsize=9)
    
    ax.set_title('Valores Nulos en Variables Numéricas',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Nº de valores faltantes')
    fig.tight_layout()
    graficos['nulos'] = img_html(fig_a_base64(fig))
    print('✅ Nulos')
else:
    graficos['nulos'] = '<p style="color:#38a169;">✅ No hay nulos en variables numéricas.</p>'
    print('✅ Sin nulos')


✅ Panorámica boxenplot
✅ Violinplots
✅ Discretas y binarias
✅ Skew vs Kurtosis (escala log)
✅ Skew vs Kurtosis (zoom)
✅ Nulos


In [5]:
# ============================================================================
# CELDA 4: GRÁFICOS — PLOTNINE + STATSMODELS
# ============================================================================

# --- 6. HISTOGRAMAS + KDE con Plotnine (grammar of graphics) ---
# Preparar datos en formato long para facet_wrap
df_long = pd.DataFrame()
medias = {}
medianas = {}

for col in continuas:
    data = df[col].dropna()
    n_val = len(data)
    label = f'{etiq(col)} (N={fmt(n_val)})'
    tmp = pd.DataFrame({'valor': data.values, 'variable': label})
    df_long = pd.concat([df_long, tmp], ignore_index=True)
    medias[label] = data.mean()
    medianas[label] = data.median()

# Líneas de referencia con colores distintos
ref_media = pd.DataFrame([
    {'variable': v, 'valor': medias[v], 'stat': 'Media'} for v in medias
])
ref_mediana = pd.DataFrame([
    {'variable': v, 'valor': medianas[v], 'stat': 'Mediana'} for v in medianas
])

from plotnine import geom_text, annotate, scale_color_manual, scale_linetype_manual

p = (ggplot(df_long, aes(x='valor'))
     + geom_histogram(aes(y=after_stat('density')),
                      bins=40, fill=COLOR_PRINCIPAL, alpha=0.5,
                      color='white', size=0.3)
     + geom_density(color=COLOR_PRINCIPAL, size=1)
     + geom_vline(aes(xintercept='valor'), data=ref_media,
                  color=COLOR_ERROR, size=0.8, linetype='solid')
     + geom_vline(aes(xintercept='valor'), data=ref_mediana,
                  color=COLOR_OK, size=0.8, linetype='dashed')
     + facet_wrap('~variable', scales='free', ncol=2)
     + labs(title='Distribuciones: Histograma + KDE (Plotnine)',
            x='Valor', y='Densidad',
            caption='Línea roja continua = Media | Línea verde discontinua = Mediana')
     + theme_minimal()
     + theme(figure_size=(12, 3.5 * ((len(continuas) + 1) // 2)),
             plot_title=element_text(size=13, weight='bold'),
             strip_text=element_text(size=9),
             plot_caption=element_text(size=9, color='gray'))
)

# Plotnine → base64
buf = io.BytesIO()
p.save(buf, format='png', dpi=150, verbose=False)
buf.seek(0)
b64 = base64.b64encode(buf.read()).decode('utf-8')
graficos['histogramas'] = img_html(b64)
print('✅ Histogramas + KDE (Plotnine)')


# --- 7. QQ-PLOTS con Statsmodels ---
n_cont = len(continuas)
ncols = 3
nrows = (n_cont + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(continuas):
    data = df[col].dropna()
    # Limitar a 5000 puntos para rendimiento
    if len(data) > 5000:
        data = data.sample(5000, random_state=42)
    
    qqplot(data, line='s', ax=axes[i], markerfacecolor=COLOR_PRINCIPAL,
           markeredgecolor='white', markersize=3, alpha=0.5)
    
    # Test Shapiro (sobre muestra)
    sample = data.sample(min(5000, len(data)), random_state=42)
    _, p_val = stats.shapiro(sample)
    normal_txt = '✅ Normal' if p_val > 0.05 else '❌ No normal'
    axes[i].set_title(f'{etiq(col)}\n{normal_txt} (p={p_val:.4f})',
                      fontsize=9)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('QQ-Plots — Test de Normalidad (Statsmodels)',
             fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
graficos['qqplots'] = img_html(fig_a_base64(fig))
print('✅ QQ-Plots (Statsmodels)')


✅ Histogramas + KDE (Plotnine)
✅ QQ-Plots (Statsmodels)


In [6]:
# ============================================================================
# CELDA 5: GENERAR HTML
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase4', modulo_activo='m03'
)

# KPIs
kpis_html = generar_kpis_html([
    {'valor': str(len(numericas)), 'titulo': 'Variables numéricas',
     'color': COLOR_PRINCIPAL},
    {'valor': str(len(continuas)), 'titulo': 'Continuas',
     'color': COLOR_OK},
    {'valor': str(len(discretas)), 'titulo': 'Discretas',
     'color': COLOR_ALERTA},
    {'valor': str(len(binarias)), 'titulo': 'Binarias',
     'color': '#805ad5'},
    {'valor': str(n_con_nulos), 'titulo': 'Con nulos',
     'color': COLOR_ERROR},
])

# Tabla estadísticas
stats_display = stats_df.copy()
stats_display.index = [etiq(c) for c in stats_display.index]
tabla_stats = (
    '<div style="overflow-x:auto;max-height:400px;overflow-y:auto;">'
    + stats_display.to_html(classes='tabla-estadisticas', border=1)
    + '</div>'
)

# Nota sobre librerías usadas
nota_librerias = (
    '<div style="background:#f0f4f8;padding:12px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-bottom:15px;font-size:0.85em;">'
    '<strong>📚 Librerías utilizadas:</strong> '
    'Seaborn (boxenplot, violinplots, barras nulos), '
    'Matplotlib (scatter skew/kurtosis, barras discretas), '
    'Plotnine (histogramas + KDE, estilo ggplot2), '
    'Statsmodels (QQ-plots y test Shapiro-Wilk). '
    'Gráficos estáticos embebidos como PNG base64.</div>'
)

# Notas de anomalías detectadas
nota_anomalias = (
    '<div style="background:#fff5f5;padding:12px;border-radius:8px;'
    'border-left:4px solid #e53e3e;margin:15px 0;font-size:0.85em;">'
    '<strong>⚠️ Anomalías detectadas:</strong><br>'
    '<strong>1. edad_entrada = 5 años</strong> (per_id_ficticio=1522852): '
    'Mujer, Rumanía, Castellón, Grado en Gestión y Adm. Pública. '
    'Fecha nacimiento registrada: 2012-12-31 (error en datos originales UJI). '
    'Probable fecha real: ~1992. Caso único (1 de 33.621 = 0,003%).<br>'
    '<strong>2. cred_superados_anio_1er &gt; 100</strong> (2.878 alumnos, 8,6%): '
    'Valores altos explicados por convalidaciones masivas (cambio de carrera, '
    'traslado de expediente). El campo incluye créditos convalidados al entrar. '
    'No es un error, pero debe tenerse en cuenta al interpretar.</div>'
)

# Construir secciones
s_nota = nota_librerias

s_panoramica = generar_seccion_html(
    'Panorámica: Distribución General (Seaborn boxenplot)',
    graficos['panoramica']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Lectura:</strong> Variables estandarizadas (media=0, std=1) '
    'para comparar escalas. Las cajas escalonadas muestran dónde se '
    'concentran los datos: más ancho = más alumnos. Puntos sueltos = outliers.</div>'
)

s_histogramas = generar_seccion_html(
    'Distribuciones: Histograma + KDE (Plotnine — ggplot2)',
    graficos['histogramas']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Leyenda:</strong> Línea roja continua = media. '
    'Línea verde discontinua = mediana. '
    'Curva azul = estimación de densidad (KDE). '
    'N indica el número de valores válidos (excluidos nulos).</div>'
)

s_violines = generar_seccion_html(
    'Violinplots — Forma de la Distribución (Seaborn)',
    graficos['violinplots']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Lectura:</strong> El ancho del violín indica la concentración '
    'de datos. Líneas internas = Q1 (percentil 25), mediana y Q3 (percentil 75). '
    'Donde el violín es más ancho hay más alumnos con ese valor.</div>'
)

s_qq = generar_seccion_html(
    'QQ-Plots — Test de Normalidad (Statsmodels)',
    graficos['qqplots']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Lectura:</strong> La línea roja = distribución normal teórica. '
    'Los puntos azules = datos reales. Si los puntos siguen la línea, '
    'la variable es normal. Desviaciones en los extremos indican colas '
    'pesadas o ligeras.<br><br>'
    '<strong>Lectura por variable:</strong><br>'
    '• <strong>Nota 1er año:</strong> La más cercana a normalidad, '
    'aunque se desvía en los extremos (notas 5 y 10).<br>'
    '• <strong>Nota acceso / selectividad:</strong> Concentración excesiva '
    'en notas bajas (5-6), curva ascendente en notas altas.<br>'
    '• <strong>Créd. superados 1er año:</strong> Muy lejos de normal. '
    'Efecto de los 0 créditos (abajo) y convalidaciones masivas (arriba).<br>'
    '• <strong>Edad entrada:</strong> Cola derecha muy larga '
    '(acceso mayores de 25/40/45 años).<br>'
    '• <strong>Orden preferencia:</strong> Completamente no normal — '
    'el 60% elige opción 1, distribución muy concentrada.<br><br>'
    '<strong>Conclusión:</strong> Ninguna variable sigue una distribución '
    'normal (test Shapiro-Wilk p&lt;0.05 en todas). Esto se tendrá en cuenta '
    'en la selección de modelos de la Fase 5.</div>'
)

s_discretas = generar_seccion_html(
    'Variables Discretas y Binarias — Frecuencias (Matplotlib)',
    graficos['discretas_binarias']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Nota:</strong> Barras verdes = binarias (0/1). '
    'Barras naranjas = discretas con pocos valores.<br><br>'
    '<strong>Hallazgos:</strong><br>'
    '• <strong>Años gap e Interrupción:</strong> 97% tiene valor 0. '
    'Muy desbalanceadas, pero el 3% con interrupción puede ser muy '
    'predictivo de abandono.<br>'
    '• <strong>Tuvo beca:</strong> 72% sí tuvo beca (1). Relativamente equilibrada.<br>'
    '• <strong>Máx. pagos:</strong> Solo valores 1, 2, 4 y 7 tienen frecuencia '
    'significativa. Los valores 3, 5, 6 son casi inexistentes (posible '
    'codificación del sistema de pagos de la UJI).<br>'
    '• <strong>Años con/sin beca:</strong> Distribuciones más repartidas, '
    'útiles para modelado.</div>'
)

# CSS para modal (reutilizable)
modal_css = """
<style>
.modal-overlay {
    display: none;
    position: fixed;
    top: 0; left: 0;
    width: 100%; height: 100%;
    background: rgba(0,0,0,0.7);
    z-index: 1000;
    justify-content: center;
    align-items: center;
}
.modal-content {
    background: white;
    border-radius: 10px;
    width: 90%;
    max-width: 1000px;
    max-height: 90%;
    overflow: auto;
    padding: 20px;
}
.modal-close {
    float: right;
    background: none;
    border: none;
    font-size: 1.5em;
    cursor: pointer;
    color: #718096;
}
.modal-close:hover { color: #e53e3e; }
.btn-zoom {
    display: inline-block;
    margin-top: 10px;
    padding: 10px 20px;
    background: #3182ce;
    color: white;
    border: none;
    border-radius: 8px;
    font-weight: 600;
    cursor: pointer;
    font-size: 0.9em;
}
.btn-zoom:hover { background: #2c5282; }
</style>
"""

s_skew = generar_seccion_html(
    'Asimetría vs Curtosis (Matplotlib)',
    modal_css
    + graficos['skew_kurtosis']
    + '<div style="text-align:center;margin-top:10px;">'
    '<button class="btn-zoom" onclick="document.getElementById(\'modal-skew-zoom\').style.display=\'flex\'">'
    '🔍 Ver zoom sin variables extremas</button></div>'
    + '<div id="modal-skew-zoom" class="modal-overlay" onclick="if(event.target==this)this.style.display=\'none\'">'
    '<div class="modal-content">'
    '<button class="modal-close" onclick="document.getElementById(\'modal-skew-zoom\').style.display=\'none\'">×</button>'
    + graficos['skew_kurtosis_zoom']
    + '</div></div>'
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Lectura:</strong> Cada punto es una variable. El color indica '
    'el nivel de curtosis:<br>'
    '• 🟢 Verde = distribución cercana a normal (curtosis ≤ 2)<br>'
    '• 🟠 Naranja = moderadamente no normal (curtosis 2-10)<br>'
    '• 🔴 Rojo = extremadamente no normal (curtosis > 10)<br><br>'
    '<strong>¿Qué es la curtosis?</strong> Mide lo "puntiaguda" que es la distribución. '
    'Años gap (curtosis=126) significa que el 97% de los datos están en un solo valor (0) '
    'y unos pocos se dispersan hasta 8. Es como un clavo: pico altísimo y casi nada alrededor. '
    'Curtosis cerca de 0 = campana normal. Negativa = distribución más plana.<br><br>'
    '<strong>¿Qué es la asimetría?</strong> Mide si la distribución está "torcida". '
    'Positiva = cola a la derecha (la mayoría tiene valores bajos). '
    'Negativa = cola a la izquierda. Cerca de 0 = simétrica.</div>'
)

s_nulos = generar_seccion_html(
    'Valores Faltantes en Variables Numéricas (Seaborn)',
    graficos['nulos']
)

s_stats = generar_seccion_html(
    'Estadísticas Descriptivas',
    tabla_stats
)

# HTML final
contenido = (
    kpis_html
    + s_nota
    + nota_anomalias
    + s_panoramica
    + s_histogramas
    + s_violines
    + s_qq
    + s_discretas
    + s_skew
    + s_nulos
    + s_stats
)

html = render_base_html(
    titulo='M03: Distribuciones Numéricas',
    icono='\U0001f4ca',
    subtitulo='Fase 4: EDA Final | TFM Abandono Universitario',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=contenido,
    notebook_nombre='f4_m03_distribuciones_num.ipynb',
    notebook_carpeta='fase4_eda',
)

ruta_html = RUTA_FASE4_HTML / 'm03_distribuciones_num.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML generado: {ruta_html}')


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m03_distribuciones_num.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m03_distribuciones_num.html


In [7]:
# ============================================================================
# CELDA 6: RESUMEN
# ============================================================================

print('=' * 70)
print('F4-M03: DISTRIBUCIONES NUMÉRICAS — RESUMEN')
print('=' * 70)
print(f'  Dataset: {n_total:,} filas × 21 columnas')
print(f'  Variables analizadas: {len(numericas)} numéricas')
print(f'    - Continuas: {len(continuas)} → {[etiq(c) for c in continuas]}')
print(f'    - Discretas: {len(discretas)} → {[etiq(c) for c in discretas]}')
print(f'    - Binarias:  {len(binarias)} → {[etiq(c) for c in binarias]}')
print(f'  Variables con nulos: {n_con_nulos}')
print(f'  Variables con outliers (IQR): {n_con_outliers}')
print(f'\n  Gráficos generados: {len(graficos)}')
print(f'  Librerías: Matplotlib, Seaborn, Plotnine, Statsmodels')
print(f'  HTML: {ruta_html}')
print('=' * 70)


F4-M03: DISTRIBUCIONES NUMÉRICAS — RESUMEN
  Dataset: 33,621 filas × 21 columnas
  Variables analizadas: 12 numéricas
    - Continuas: 6 → ['Créd. superados 1er año', 'Edad entrada', 'Nota 1er año', 'Nota acceso', 'Nota selectividad', 'Orden preferencia']
    - Discretas: 4 → ['Años gap', 'Años sin beca', 'Máx. pagos', 'Años con beca']
    - Binarias:  2 → ['Interrupción', 'Tuvo beca']
  Variables con nulos: 5
  Variables con outliers (IQR): 6

  Gráficos generados: 8
  Librerías: Matplotlib, Seaborn, Plotnine, Statsmodels
  HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m03_distribuciones_num.html
